# Aula 6 — Avaliação de classificadores de texto

**Por que acurácia não conta a história toda**

Na Aula 5, construímos nosso primeiro classificador e usamos acurácia como primeira medida.

Agora vamos abrir essa caixa:

```text
predições
→ matriz de confusão
→ TP / FP / FN
→ precision / recall / F1
→ decisão de negócio
```

A meta não é apenas chamar `classification_report`, mas conseguir conferir de onde as métricas vêm.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar por que acurácia pode ser insuficiente;
- interpretar uma matriz de confusão;
- calcular TP, FP, FN e TN para uma classe em abordagem one-vs-rest;
- derivar precision, recall e F1 manualmente;
- conferir os cálculos com `scikit-learn`;
- interpretar métricas por classe;
- comparar classificadores com mesma acurácia e perfis de erro diferentes;
- relacionar métricas ao custo real dos erros.


## 📘 Glossário da aula

Conceitos centrais desta aula: **[matriz de confusão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#matriz-de-confusão) · [precisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#precisão) · [recall](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#recall) · [F1-score](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#f1-score) · [macro average](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#macro-average) · [weighted average](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#weighted-average)**.

Use o Glossário Vivo quando quiser revisar uma definição, conferir o termo técnico em inglês ou retomar a relação entre conceitos.

- [Glossário PT-BR](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md)
- [Glossary EN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.en.md)

> O notebook explica o necessário para seguir a aula; o glossário serve para consolidar e aprofundar o vocabulário técnico.


## Antes de começar — trabalhe na sua própria cópia

Crie sua cópia do notebook no Kaggle antes de executar ou modificar qualquer célula.


## 2. O limite da acurácia

Acurácia responde:

> **qual proporção total das previsões estava correta?**

Ela não responde:

- em quais classes o modelo errou;
- qual classe deixou de recuperar;
- qual classe gerou falsos alarmes;
- se dois modelos com a mesma acurácia erram de formas diferentes.

Nesta aula, vamos produzir evidência para essa afirmação.


## 3. Um pequeno cenário controlado

Vamos simular resultados de um classificador em três classes.


In [ ]:
y_true = [
    "duvida", "duvida", "duvida", "duvida",
    "reclamacao", "reclamacao", "reclamacao", "reclamacao",
    "elogio", "elogio", "elogio", "elogio",
]

y_pred = [
    "duvida", "duvida", "reclamacao", "duvida",
    "reclamacao", "duvida", "reclamacao", "reclamacao",
    "elogio", "elogio", "duvida", "elogio",
]


## 4. Matriz de confusão

A matriz de confusão mostra como as classes reais foram confundidas com as classes previstas.

Execute a próxima célula.


In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

labels = ["duvida", "reclamacao", "elogio"]
cm = confusion_matrix(y_true, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"real_{label}" for label in labels],
    columns=[f"prev_{label}" for label in labels],
)

cm_df


### Como ler

- diagonal principal = acertos;
- fora da diagonal = erros;
- linhas = classe real;
- colunas = classe prevista.

**Checkpoint 1:** identifique qual classe foi confundida com `duvida`.


## 5. Uma classe de cada vez: one-vs-rest

Para entender precision e recall em um problema multiclasse, podemos escolher uma classe e perguntar:

> “`reclamacao`” versus “não `reclamacao`”.

Para essa classe:

- **TP**: era reclamação e foi prevista como reclamação;
- **FP**: não era reclamação, mas foi prevista como reclamação;
- **FN**: era reclamação, mas recebeu outra classe;
- **TN**: não era reclamação e não foi prevista como reclamação.

Vamos calcular esses valores diretamente das listas de rótulos.


In [ ]:
target_class = "reclamacao"

tp = sum(
    true == target_class and pred == target_class
    for true, pred in zip(y_true, y_pred)
)

fp = sum(
    true != target_class and pred == target_class
    for true, pred in zip(y_true, y_pred)
)

fn = sum(
    true == target_class and pred != target_class
    for true, pred in zip(y_true, y_pred)
)

tn = sum(
    true != target_class and pred != target_class
    for true, pred in zip(y_true, y_pred)
)

manual_counts = pd.Series({
    "TP": tp,
    "FP": fp,
    "FN": fn,
    "TN": tn,
})

display(manual_counts)

assert tp + fp + fn + tn == len(y_true)


## 6. Derivando precision, recall e F1

Agora as fórmulas deixam de ser abstrações:

[
Precision = \frac{TP}{TP + FP}
]

[
Recall = \frac{TP}{TP + FN}
]

[
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
]

Antes de executar a próxima célula, preveja:

- aumentar FP piora qual métrica diretamente?
- aumentar FN piora qual métrica diretamente?


In [ ]:
manual_precision = tp / (tp + fp) if (tp + fp) else 0.0
manual_recall = tp / (tp + fn) if (tp + fn) else 0.0

manual_f1 = (
    2 * manual_precision * manual_recall
    / (manual_precision + manual_recall)
    if (manual_precision + manual_recall)
    else 0.0
)

manual_metrics = pd.Series({
    "precision": manual_precision,
    "recall": manual_recall,
    "f1": manual_f1,
})

display(manual_metrics.round(3))


### Conferindo com a biblioteca

Uma biblioteca é útil, mas nesta aula ela funciona como **verificação independente** do cálculo que acabamos de fazer.


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

sk_precision, sk_recall, sk_f1, support = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[target_class],
        average=None,
        zero_division=0,
    )
)

verification = pd.DataFrame([
    {
        "source": "manual",
        "precision": manual_precision,
        "recall": manual_recall,
        "f1": manual_f1,
    },
    {
        "source": "sklearn",
        "precision": sk_precision[0],
        "recall": sk_recall[0],
        "f1": sk_f1[0],
    },
])

display(verification.round(3))

assert np.allclose(
    verification.loc[0, ["precision", "recall", "f1"]].astype(float),
    verification.loc[1, ["precision", "recall", "f1"]].astype(float),
)

print("Cálculo manual confere com sklearn.")


## 7. O que precision, recall e F1 realmente medem

Agora podemos interpretar pelas contagens:

### Precision

```text
entre tudo que chamei de reclamação,
quanto realmente era reclamação?
```

Sensível a **falsos positivos**.

### Recall

```text
entre todas as reclamações reais,
quantas consegui recuperar?
```

Sensível a **falsos negativos**.

### F1

Combina precision e recall por média harmônica.

Por isso, F1 cai quando uma das duas métricas fica muito baixa.


## 8. Classification report — agora como resumo, não como caixa-preta

Depois de derivar uma classe manualmente, podemos usar `classification_report` para repetir o mesmo raciocínio em todas as classes.


In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    y_true,
    y_pred,
    labels=labels,
    digits=3,
)

print(report)


### O que interpretar

Observe as métricas por classe, não apenas a média geral.

Uma classe pode ter recall baixo mesmo quando a acurácia global parece aceitável.

**Checkpoint 2:** procure a classe com pior F1-score e pense em que tipo de erro está ocorrendo.


## 9. Same Accuracy Lab — mesmo número, erros diferentes

Vamos comparar dois classificadores sintéticos com **a mesma acurácia total**, mas distribuições de erro diferentes.

A pergunta é:

> a acurácia sozinha é suficiente para escolher entre eles?


In [ ]:
from sklearn.metrics import accuracy_score

same_true = [
    "reclamacao", "reclamacao", "reclamacao", "reclamacao",
    "duvida", "duvida", "duvida", "duvida",
    "elogio", "elogio", "elogio", "elogio",
]

pred_model_a = [
    "reclamacao", "reclamacao", "reclamacao", "reclamacao",
    "duvida", "duvida", "elogio", "elogio",
    "elogio", "elogio", "duvida", "duvida",
]

pred_model_b = [
    "reclamacao", "duvida", "duvida", "duvida",
    "duvida", "duvida", "duvida", "duvida",
    "elogio", "elogio", "elogio", "reclamacao",
]

print("Accuracy A:", accuracy_score(same_true, pred_model_a))
print("Accuracy B:", accuracy_score(same_true, pred_model_b))


In [ ]:
comparison_rows = []

for name, pred in [
    ("model_a", pred_model_a),
    ("model_b", pred_model_b),
]:
    p, r, f, _ = precision_recall_fscore_support(
        same_true,
        pred,
        labels=["reclamacao"],
        average=None,
        zero_division=0,
    )

    comparison_rows.append({
        "model": name,
        "accuracy": accuracy_score(same_true, pred),
        "reclamacao_precision": p[0],
        "reclamacao_recall": r[0],
        "reclamacao_f1": f[0],
    })

same_accuracy_comparison = pd.DataFrame(comparison_rows)
display(same_accuracy_comparison.round(3))


### Interpretação

Os dois modelos podem acertar a mesma quantidade total e ainda tratar `reclamacao` de maneira muito diferente.

Se perder reclamações críticas for caro, essa diferença importa.

Portanto:

```text
same accuracy
≠
same error profile
≠
same business utility
```

Esse é o ponto que será aprofundado mais tarde quando o TIL introduzir utility, custo e risco de forma mais ampla.


## 10. Métrica e custo do erro

As métricas não escolhem prioridades sozinhas.

### Cenário A — triagem de reclamações críticas

Perder uma reclamação importante pode ser pior que gerar um falso alerta.

Nesse caso, **recall** pode receber maior peso.

### Cenário B — alerta automático que dispara ação cara

Falsos positivos podem causar operações desnecessárias.

Nesse caso, **precision** pode receber maior peso.

A métrica deve refletir o custo do erro, não apenas conveniência estatística.


## 11. Macro, weighted e micro

Depois de entender a métrica por classe, podemos agregar:

- **macro average**: cada classe pesa igualmente;
- **weighted average**: classes pesam conforme seu suporte;
- **micro average**: decisões são agregadas antes do cálculo.

Essas médias respondem perguntas diferentes.

Não escolha entre elas apenas pelo maior número.


## 12. Exercício — calcule antes de chamar sklearn

Use:

```python
exercise_true = [
    'positivo', 'positivo', 'positivo',
    'negativo', 'negativo', 'negativo'
]

exercise_pred = [
    'positivo', 'negativo', 'positivo',
    'negativo', 'negativo', 'positivo'
]
```

Para a classe `positivo`:

1. calcule TP, FP, FN e TN manualmente;
2. derive precision;
3. derive recall;
4. derive F1;
5. confira com `precision_recall_fscore_support`;
6. explique qual tipo de erro reduz o recall observado.


In [ ]:
# Escreva sua solução aqui.

exercise_true = ['positivo', 'positivo', 'positivo', 'negativo', 'negativo', 'negativo']
exercise_pred = ['positivo', 'negativo', 'positivo', 'negativo', 'negativo', 'positivo']

# Continue a partir daqui.


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q6.hint()` e `q6.solution()`.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))


q6 = TILExercise(
    hint_text=(
        "Trate 'positivo' como classe-alvo e conte cada par "
        "(real, previsto) antes de aplicar as fórmulas."
    ),
    solution_text=(
        "Uma solução forte mostra explicitamente TP/FP/FN/TN, "
        "calcula precision = TP/(TP+FP), recall = TP/(TP+FN), "
        "F1 pela média harmônica e só depois usa sklearn para conferir."
    ),
)

print("Tente resolver antes de usar q6.hint() ou q6.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q6.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a solução.
# q6.solution()


## 13. Reprodutibilidade

- Python;
- `scikit-learn`, `pandas` e `numpy`;
- CPU;
- Internet OFF;
- dados definidos no notebook;
- nenhuma métrica numérica fixa no markdown;
- cálculos manuais conferidos com biblioteca.


## 14. Síntese

A cadeia de evidência desta aula é:

```text
y_true + y_pred
→ confusion matrix
→ TP / FP / FN / TN
→ precision / recall / F1
→ comparação por classe
→ custo do erro
```

E o laboratório de mesma acurácia mostrou:

```text
same accuracy
≠
same error profile
```

### Ideia principal

> **Uma métrica só é útil quando sabemos de quais erros ela é construída e por que esses erros importam para o problema.**



---

## Continue no TIL

← **[Anterior: Aula 5 — Primeiro classificador de textos](https://www.kaggle.com/code/pedrogentil/til-05-first-text-classifier)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-course)** &nbsp;&nbsp;|&nbsp;&nbsp; **[Próxima: Aula 7 — Seleção de modelos e tuning](https://www.kaggle.com/code/pedrogentil/til-07-model-selection-and-tuning)** →
